#02 - EDA y limpieza de datos

Preparación: cargamos el dataset raw, se realiza la limpieza y se organiza el dataset limpio para el modelado y el dashboard

In [1]:
# importamos la base de datos cruda de icfes.

import pandas as pd

df = pd.read_csv("../data/raw/saber11_2025-1.csv")
print(df.shape)
df.head()

(1313, 85)


,periodo,estu_consecutivo,estu_estudiante,estu_tipodocumento,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_cod_dane_establecimiento,cole_cod_dane_sede,...,percentil_lectura_critica,percentil_matematicas,percentil_sociales_ciudadanas,punt_c_naturales,punt_global,punt_ingles,punt_lectura_critica,punt_matematicas,punt_sociales_ciudadanas,estu_grupoetnia
0,20251,SB11202510069266,ESTUDIANTE,TI,RURAL,S,B,ACADÉMICO,305001003688,305001003688,...,66,75,77,63,335,81.0,65,69,66,NINGUNO
1,20251,SB11202510003099,ESTUDIANTE,TI,URBANA,NaN,B,ACADÉMICO,305001019681,305001019681,...,39,52,63,59,302,83.0,56,59,60,NINGUNO
2,20251,SB11202510054057,ESTUDIANTE,CC,URBANO,N,A,ACADÉMICO,105088002918,105088003175,...,35,28,27,34,225,52.0,54,47,43,NINGUNO
3,20251,SB11202510053170,ESTUDIANTE,TI,RURAL,NaN,A,NaN,305360800046,305360800046,...,5,22,25,35,197,41.0,36,44,42,NINGUNO
4,20251,SB11202510053800,ESTUDIANTE,TI,URBANO,N,A,ACADÉMICO,105088002918,105088003175,...,26,32,12,39,218,45.0,50,50,35,NINGUNO


In [2]:
#Selección de las columnas que se van a tener
#presentes.

cols = [
    "punt_global",                                              # para derivar el objetivo
    "estu_genero", "fami_estratovivienda", "fami_tieneinternet",
    "fami_educacionmadre", "fami_educacionpadre",               # features
    "cole_mcpio_ubicacion", "cole_cod_mcpio_ubicacion",         # municipio: nombre + código
    "cole_naturaleza", "cole_area_ubicacion",                   # contexto
]
df = df[cols]
df.shape

(1313, 10)

In [3]:
#diagnostico incial de los datos.

print("Shape:", df.shape)
print("\nNulos:\n", df.isnull().sum())
print("\nDuplicados:", df.duplicated().sum())

Shape: (1313, 10)

Nulos:
 punt_global                   0
estu_genero                   0
fami_estratovivienda        193
fami_tieneinternet          191
fami_educacionmadre         193
fami_educacionpadre         191
cole_mcpio_ubicacion          0
cole_cod_mcpio_ubicacion      0
cole_naturaleza               0
cole_area_ubicacion           0
dtype: int64

Duplicados: 25


In [4]:
# Corrección de Urbano/Urbano


df["cole_area_ubicacion"] = df["cole_area_ubicacion"].replace({"URBANO": "URBANA"})
df["cole_area_ubicacion"].value_counts()


cole_area_ubicacion
URBANA    863
RURAL     450
Name: count, dtype: int64

In [5]:
#Eliminación de datos nulos

features = ["estu_genero", "fami_estratovivienda", "fami_tieneinternet",
            "fami_educacionmadre", "fami_educacionpadre"]
df = df.dropna(subset=features)
print("Filas tras limpieza:", df.shape[0])

Filas tras limpieza: 1109


In [6]:
#construcción de variable objetivo desempeño.

df["desempeno_global"] = pd.cut(df["punt_global"], bins=[-1, 220, 320, 500],
                                labels=["Bajo", "Medio", "Alto"])
df["desempeno_global"].value_counts()

desempeno_global
Alto     435
Medio    415
Bajo     259
Name: count, dtype: int64

In [7]:
#guardado del dataset limpio.

import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/dataset_limpio.csv", index=False)

prueba = pd.read_csv("../data/processed/dataset_limpio.csv")
print("Guardado OK:", prueba.shape)
prueba.head()

Guardado OK: (1109, 11)


,punt_global,estu_genero,fami_estratovivienda,fami_tieneinternet,fami_educacionmadre,fami_educacionpadre,cole_mcpio_ubicacion,cole_cod_mcpio_ubicacion,cole_naturaleza,cole_area_ubicacion,desempeno_global
0,335,F,Estrato 5,Si,Postgrado,Educación profesional completa,ENVIGADO,5266,NO OFICIAL,RURAL,Alto
1,302,M,Estrato 5,Si,Postgrado,Postgrado,ENVIGADO,5266,NO OFICIAL,URBANA,Medio
2,197,M,Estrato 2,Si,Secundaria (Bachillerato) completa,Primaria incompleta,ITAGÜÍ,5360,NO OFICIAL,RURAL,Bajo
3,356,F,Estrato 6,Si,Postgrado,Postgrado,ENVIGADO,5266,NO OFICIAL,URBANA,Alto
4,225,M,Estrato 2,Si,Secundaria (Bachillerato) completa,Secundaria (Bachillerato) completa,MEDELLÍN,5001,NO OFICIAL,URBANA,Medio
